# B2.13 · Building the threat-modelling harness

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.12 · Building the DAST and exploitation harness](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**.

| | |
|---|---|
| Open-source tooling | OWASP Threat Dragon, Trivy |
| Open-weight models | GLM-5.2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A threat model is usually produced once, by hand, in a workshop, for a system
that changes every release. By the second sprint it describes something that no
longer exists — and nobody notices, because nothing re-reads it.

The threat-modelling harness makes it a **derived artefact**: generated from
architecture, IaC, code and data flows, regenerated on every release, and — the
part that carries the value — **diffed between versions**.

The diff is the product. A threat model tells you what could go wrong; a threat
model *diff* tells you what this pull request just introduced, which is a
question a human can act on within a review cycle.

That reframes human judgement too. Reviewing a hundred threats a quarter is a
workshop nobody attends. Reviewing the four threats that appeared since Tuesday
is a checkpoint, and a checkpoint is something an engineering process can carry.

For the diff to mean anything the generation must be **deterministic**: same
input, same output, byte for byte. Two runs that disagree about unchanged code
produce a diff full of noise, and a noisy diff is one people stop reading.

## 2 · Derive the model from the system, not from a workshop

In [ ]:
ARCH_V1 = {
 "components": {"web": {"trust": 0, "exposes": ["/report", "/upload"]},
                "api": {"trust": 1, "exposes": ["/internal/report"]},
                "db":  {"trust": 2, "exposes": []}},
 "flows": [("web", "api"), ("api", "db")],
 "sinks": {"db": "database", "api": "filesystem"},
}

def derive(arch):
    """Deterministic by construction: everything sorted, nothing from a set."""
    threats = []
    for a, b in sorted(arch["flows"]):
        ta, tb = arch["components"][a]["trust"], arch["components"][b]["trust"]
        if ta < tb and b in arch["sinks"]:
            for entry in sorted(arch["components"][a]["exposes"]) or ["-"]:
                threats.append({
                    "entry": entry, "path": f"{a} -> {b}",
                    "resource": arch["sinks"][b],
                    "crossing": f"trust {ta} -> {tb}",
                    "score": (tb - ta) * (2 if arch["sinks"][b] == "database" else 1)})
    return sorted(threats, key=lambda t: (-t["score"], t["entry"], t["path"]))

tm1 = derive(ARCH_V1)
print(f"{'entry':22s}{'path':16s}{'resource':12s}{'crossing':18s}score")
for t in tm1:
    print(f"{t['entry']:22s}{t['path']:16s}{t['resource']:12s}{t['crossing']:18s}{t['score']}")
print(f"\n{len(tm1)} threats derived")

## 3 · Change one thing, and diff

In [ ]:
import copy
ARCH_V2 = copy.deepcopy(ARCH_V1)
ARCH_V2["components"]["web"]["exposes"].append("/admin/export")   # one new route
ARCH_V2["components"]["worker"] = {"trust": 0, "exposes": ["queue"]}
ARCH_V2["flows"].append(("worker", "db"))                          # and one new flow

tm2 = derive(ARCH_V2)

def key(t): return (t["entry"], t["path"], t["resource"])
def diff(before, after):
    a = {key(t): t for t in after}
    b = {key(t): t for t in before}
    return ({k: a[k] for k in sorted(a.keys() - b.keys())},
            {k: b[k] for k in sorted(b.keys() - a.keys())})

new, gone = diff(tm1, tm2)
print(f"threats before {len(tm1)}  ->  after {len(tm2)}")
print(f"\nNEW ({len(new)}):")
for k in new:
    t = new[k]
    print(f"   score {t['score']}  {t['entry']:22s}{t['path']:16s}{t['resource']}")
print(f"REMOVED ({len(gone)}): {list(gone) or 'none'}")
print()
print("Two lines of infrastructure change. Three new threats, one of them from")
print("a component nobody mentioned in the pull request description.")
assert new and not gone

## 4 · Where it breaks — a diff you cannot trust\n\nRegenerate the *identical* architecture with a generator that iterates a set instead of a sorted sequence.

In [ ]:
def derive_unstable(arch, seed):
    """Same logic, but the ordering comes from set iteration."""
    import random
    threats = derive(arch)
    rng = random.Random(seed)
    rng.shuffle(threats)                 # stands in for hash-order instability
    return threats

runs = [derive_unstable(ARCH_V1, s) for s in range(3)]
identical = all(r == runs[0] for r in runs)
print(f"three regenerations of UNCHANGED architecture identical: {identical}")

n1, g1 = diff(runs[0], runs[1])
print(f"diff between two runs of the same input: {len(n1)} new, {len(g1)} removed")
print("   (content-keyed, so the noise does not show here...)")

# but a line-oriented diff, which is what a reviewer actually reads:
lines = [[f"{t['entry']} {t['path']} {t['score']}" for t in r] for r in runs]
noisy = sum(1 for a, b in zip(lines[0], lines[1]) if a != b)
print(f"   line-by-line, {noisy} of {len(lines[0])} lines differ on unchanged input")
print()
print("Nothing changed and the review shows churn. Reviewers learn within about")
print("two sprints that the diff is noise, and then they stop reading it - which")
print("costs you the one output of this harness that was worth having.")
assert noisy > 0

## 5 · The control — determinism, then the checkpoint

In [ ]:
stable = [derive(ARCH_V1) for _ in range(5)]
print(f"five regenerations identical: {all(s == stable[0] for s in stable)}")

def review_load(before, after, per_threat_minutes=8):
    new, gone = diff(before, after)
    return {"total_threats": len(after),
            "needs_review": len(new),
            "workshop_minutes": len(after) * per_threat_minutes,
            "checkpoint_minutes": len(new) * per_threat_minutes}

r = review_load(tm1, tm2)
print(f"\nthreats in the model      : {r['total_threats']}")
print(f"threats needing review    : {r['needs_review']}")
print(f"review the whole model    : {r['workshop_minutes']} minutes")
print(f"review only the diff      : {r['checkpoint_minutes']} minutes")
print()
print("The whole model is a workshop nobody attends. The diff is a checkpoint")
print("that fits in a pull request, and it contains the same new information.")
assert all(s == stable[0] for s in stable)

## 6 · Verify — the diff catches what the description omitted

In [ ]:
PR_DESCRIPTION = "add an admin export route"
mentioned = "/admin/export"
surprises = [new[k] for k in new if mentioned not in new[k]["entry"]]
print(f"pull request says      : {PR_DESCRIPTION!r}")
print(f"threats it introduced  : {len(new)}")
print(f"not implied by the text: {len(surprises)}")
for t in surprises:
    print(f"   {t['path']:16s}{t['resource']:12s}{t['crossing']}")
print()
print("The worker flow is a legitimate change made for a good reason. It is")
print("also a second untrusted path to the database, and the sentence describing")
print("the pull request does not contain it.")
assert surprises

## What you just proved

A threat model derived from architecture yields threats sorted deterministically. Two lines of infrastructure change introduce three new threats, one from a component the pull request description never mentions. An unstable generator produces a diff full of churn on unchanged input, and the deterministic one regenerates identically five times — turning a whole-model workshop into a review of only what changed.

## Your turn

Generate a threat model for your service twice, without changing anything, and diff the two. If they differ, the diff you were planning to review on every release was never going to work.

---

**Next → [B2.14 · Building the pentest harness](https://spbreed.github.io/cyber-commons/lessons/B2.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*